In [1]:
from importlib.metadata import entry_points

import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

# TorchText，TorchVision 和 TorchAudio 是 PyTorch 生态系统中的三个重要库，分别用于处理文本、图像和音频数据。
# 它们提供了丰富的数据集、预处理工具和模型，方便用户进行

In [2]:
# 加载 FashionMNIST 数据集
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor()
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor()
)

In [3]:
batch_size = 64
train_dataloader = DataLoader(training_data,batch_size=batch_size)
test_dataloader = DataLoader(test_data,batch_size=batch_size)

for X,y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}, dtype: {X.dtype}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    # print(f"X: {X}")
    # print(f"y: {y}")
    break

Shape of X [N, C, H, W]: torch.Size([64, 1, 28, 28]), dtype: torch.float32
Shape of y: torch.Size([64]) torch.int64


In [4]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using device: {device}")

class NeuralNetwork(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.flatten = nn.Flatten()
        self.liner_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )
    def forward(self, x):
        x = self.flatten(x)
        logits = self.liner_relu_stack(x)
        return logits
    
model = NeuralNetwork().to(device)
print(model)

Using device: cuda
NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (liner_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [5]:
# 优化模型参数
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(),lr=1e-3)

In [6]:
def train(dataloader,model,loss_fn,optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch,(x,y) in enumerate(dataloader):
        X,y = x.to(device),y.to(device)
        pred = model(X)
        loss = loss_fn(pred,y)
        
        # 反向传播
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss,current = loss.item(),(batch+1) * len(X)
            print(f"loss:{loss:>7f} [{current:5d}/{size:>5d}]")

In [7]:
def test(dataloader,model,loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss ,correct = 0,0
    with torch.no_grad():
        for batch,(X,y) in enumerate(dataloader):
            X,y = X.to(device),y.to(device)
            pred = model(X)
            test_loss = loss_fn(pred,y)
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error:\n Accuracy :{(100*correct):>0.1f}%,Avg loss:{test_loss:>8f} \n")


In [8]:
epochs = 20
for e in range(epochs):
    print(f"Epoch {e+1} \n ----------------------------")
    train(train_dataloader,model,loss_fn,optimizer)
    test(test_dataloader,model,loss_fn)
print('Done')

Epoch 1 
 ----------------------------
loss:2.294574 [   64/60000]
loss:2.288648 [ 6464/60000]
loss:2.272960 [12864/60000]
loss:2.271839 [19264/60000]
loss:2.244869 [25664/60000]
loss:2.220827 [32064/60000]
loss:2.228282 [38464/60000]
loss:2.200797 [44864/60000]
loss:2.193049 [51264/60000]
loss:2.163853 [57664/60000]
Test Error:
 Accuracy :49.7%,Avg loss:0.013943 

Epoch 2 
 ----------------------------
loss:2.162585 [   64/60000]
loss:2.158929 [ 6464/60000]
loss:2.107229 [12864/60000]
loss:2.122915 [19264/60000]
loss:2.074388 [25664/60000]
loss:2.008520 [32064/60000]
loss:2.038908 [38464/60000]
loss:1.965694 [44864/60000]
loss:1.964248 [51264/60000]
loss:1.891675 [57664/60000]
Test Error:
 Accuracy :58.0%,Avg loss:0.012380 

Epoch 3 
 ----------------------------
loss:1.924324 [   64/60000]
loss:1.901045 [ 6464/60000]
loss:1.788135 [12864/60000]
loss:1.824375 [19264/60000]
loss:1.721089 [25664/60000]
loss:1.654163 [32064/60000]
loss:1.683072 [38464/60000]
loss:1.582893 [44864/60000]
l

In [9]:
torch.save(model.state_dict(),"model.pth")
print("Saved PyTorch Model State to model.pth")

Saved PyTorch Model State to model.pth


In [25]:
classes = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

model.eval()
x, y = test_data[0][0],test_data[0][1]
with torch.no_grad():
    x = x.to(device)
    pred = model(x)
    print(f"pred:{pred}")
    print(f"y:{y}")
    predicted , actual = classes[pred[0].argmax(-1)],classes[y]
    print(f"Predicted :{predicted} \nActual: {actual}")

pred:tensor([[-3.2364, -5.0163, -3.7348, -2.5291, -3.3497,  6.0988, -3.1291,  6.1195,
          2.6432,  6.8519]], device='cuda:0')
y:9
Predicted :Ankle boot 
Actual: Ankle boot


4